# 🐍 Day 7: Mini-Project - Plugin System

- Extensible plugin architecture
- Plugin discovery
- Plugin interface
- Complete runnable system

## Project Overview

We'll build a plugin system that:
- Defines a Plugin base class/interface
- Auto-discovers plugins (via registry)
- Allows plugins to register and be invoked
- Supports plugin-specific configuration

## Step 1: Plugin Base Class

Define the interface all plugins must implement.

In [ ]:
from abc import ABC, abstractmethod

class Plugin(ABC):
    @property
    @abstractmethod
    def name(self) -> str:
        pass

    @abstractmethod
    def run(self, *args, **kwargs):
        pass

    def __str__(self):
        return self.name

## Step 2: Plugin Registry

Use `__init_subclass__` to auto-register plugins.

In [ ]:
class PluginRegistry:
    _plugins = {}

    @classmethod
    def register(cls, plugin_class):
        if issubclass(plugin_class, Plugin):
            instance = plugin_class()
            cls._plugins[instance.name] = instance
        return plugin_class

    @classmethod
    def get(cls, name):
        return cls._plugins.get(name)

    @classmethod
    def list_plugins(cls):
        return list(cls._plugins.keys())

## Step 3: Concrete Plugins

Implement a few plugins.

In [ ]:
class GreetPlugin(Plugin):
    @property
    def name(self):
        return "greet"

    def run(self, name="World", **kwargs):
        return f"Hello, {name}!"

class UppercasePlugin(Plugin):
    @property
    def name(self):
        return "uppercase"

    def run(self, text="", **kwargs):
        return text.upper()

class ReversePlugin(Plugin):
    @property
    def name(self):
        return "reverse"

    def run(self, text="", **kwargs):
        return text[::-1]

# Register plugins
PluginRegistry.register(GreetPlugin)
PluginRegistry.register(UppercasePlugin)
PluginRegistry.register(ReversePlugin)

## Step 4: Plugin Manager

Orchestrate plugin execution.

In [ ]:
class PluginManager:
    def __init__(self):
        self.registry = PluginRegistry

    def run(self, plugin_name, *args, **kwargs):
        plugin = self.registry.get(plugin_name)
        if plugin is None:
            raise ValueError(f"Unknown plugin: {plugin_name}")
        return plugin.run(*args, **kwargs)

    def run_all(self, *args, **kwargs):
        results = {}
        for name in self.registry.list_plugins():
            plugin = self.registry.get(name)
            results[name] = plugin.run(*args, **kwargs)
        return results

    def pipeline(self, plugin_names, initial_data):
        data = initial_data
        for name in plugin_names:
            plugin = self.registry.get(name)
            if plugin:
                data = plugin.run(text=data) if "text" in str(plugin.run.__code__.co_varnames) else plugin.run(data)
        return data

## Step 5: Simpler Pipeline

Pipeline that passes data through plugins. Simplify the pipeline logic.

In [ ]:
class PluginManager:
    def __init__(self):
        self.registry = PluginRegistry

    def run(self, plugin_name, *args, **kwargs):
        plugin = self.registry.get(plugin_name)
        if plugin is None:
            raise ValueError(f"Unknown plugin: {plugin_name}")
        return plugin.run(*args, **kwargs)

    def pipeline(self, plugin_names, text):
        result = text
        for name in plugin_names:
            plugin = self.registry.get(name)
            if plugin:
                result = plugin.run(text=result)
        return result

manager = PluginManager()
print(manager.run("greet", name="Alice"))
print(manager.run("uppercase", text="hello"))
print(manager.pipeline(["uppercase", "reverse"], "hello"))

## Step 6: Plugin with Config

Add optional configuration to plugins.

In [ ]:
class PrefixPlugin(Plugin):
    def __init__(self, prefix=">> "):
        self._prefix = prefix

    @property
    def name(self):
        return "prefix"

    def run(self, text="", **kwargs):
        return self._prefix + text

PluginRegistry._plugins["prefix"] = PrefixPlugin(">> ")

manager = PluginManager()
print(manager.run("prefix", text="Hello"))

## Step 7: Complete System

Full plugin system with discovery and execution.

In [ ]:
from abc import ABC, abstractmethod

class Plugin(ABC):
    @property
    @abstractmethod
    def name(self) -> str:
        pass

    @abstractmethod
    def run(self, *args, **kwargs):
        pass

class PluginRegistry:
    _plugins = {}

    @classmethod
    def register(cls, plugin):
        if isinstance(plugin, type) and issubclass(plugin, Plugin):
            p = plugin()
            cls._plugins[p.name] = p
        elif isinstance(plugin, Plugin):
            cls._plugins[plugin.name] = plugin
        return plugin

    @classmethod
    def get(cls, name):
        return cls._plugins.get(name)

    @classmethod
    def list_plugins(cls):
        return list(cls._plugins.keys())

class GreetPlugin(Plugin):
    @property
    def name(self): return "greet"
    def run(self, name="World", **kw): return f"Hello, {name}!"

class UppercasePlugin(Plugin):
    @property
    def name(self): return "uppercase"
    def run(self, text="", **kw): return text.upper()

class ReversePlugin(Plugin):
    @property
    def name(self): return "reverse"
    def run(self, text="", **kw): return text[::-1]

PluginRegistry.register(GreetPlugin)
PluginRegistry.register(UppercasePlugin)
PluginRegistry.register(ReversePlugin)

class PluginManager:
    def run(self, name, *a, **kw):
        p = PluginRegistry.get(name)
        if not p: raise ValueError(f"Unknown: {name}")
        return p.run(*a, **kw)

    def pipeline(self, names, text):
        r = text
        for n in names:
            p = PluginRegistry.get(n)
            if p: r = p.run(text=r)
        return r

## Step 8: Demo

Run the plugin system.

In [ ]:
m = PluginManager()
print("Plugins:", PluginRegistry.list_plugins())
print(m.run("greet", name="Alice"))
print(m.run("uppercase", text="hello world"))
print(m.run("reverse", text="hello"))
print("Pipeline uppercase -> reverse:", m.pipeline(["uppercase", "reverse"], "hello"))

## Extensions to Try

- Load plugins from a directory (importlib)
- Plugin dependencies and ordering
- Plugin enable/disable
- Async plugins
- Plugin hooks (before_run, after_run)

## Key Takeaways

- Abstract base class defines plugin contract
- Registry for discovery and lookup
- Pipeline for chaining plugins
- Easy to add new plugins without modifying core

## 🎉 Month 3 Complete!

You've completed Month 3: decorators, metaclasses, testing, concurrency, design patterns, and performance. Congratulations!